# 🔬 01 - Exploratory Data Analysis (EDA) on HAM10000 Dataset

This notebook inspects the **HAM10000** (*Human Against Machine with 10,000 training images*) dataset:
- Class frequency distribution & severe class imbalance analysis
- Demographic breakdowns (patient age, gender, anatomical lesion sites)
- Visual inspection of representative dermoscopy lesion samples per class

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Resolve repository paths dynamically
NOTEBOOK_DIR = Path(os.getcwd())
REPO_ROOT = NOTEBOOK_DIR.parent.parent
DATA_DIR = REPO_ROOT / "ml_engine" / "data"

print(f"📁 Repository Root: {REPO_ROOT}")
print(f"📁 Data Directory:  {DATA_DIR}")

## 1. Load Dataset Metadata
Reads `HAM10000_metadata.csv` containing diagnostic ground truth, patient demographics, and lesion IDs.

In [ ]:
# Define 7 Diagnostic Classes and medical names
CLASS_MAP = {
    'akiec': 'Actinic Keratoses (Solar Keratoses)',
    'bcc':   'Basal Cell Carcinoma',
    'bkl':   'Benign Keratosis-like Lesions',
    'df':    'Dermatofibroma',
    'mel':   'Melanoma',
    'nv':    'Melanocytic Nevi',
    'vasc':  'Vascular Lesions'
}

# Look for metadata CSV in data folder
metadata_candidates = list(DATA_DIR.rglob("HAM10000_metadata.csv"))
if metadata_candidates:
    metadata_path = metadata_candidates[0]
    df = pd.read_csv(metadata_path)
    df["dx_full"] = df["dx"].map(CLASS_MAP)
    print(f"✅ Metadata loaded successfully: {len(df)} records")
    display(df.head())
else:
    print("⚠️ HAM10000_metadata.csv not found in ml_engine/data. Please run download script first.")

## 2. Class Distribution & Imbalance Analysis
Examines the diagnostic target class distribution to determine necessary class-weight compensations.

In [ ]:
if 'df' in locals():
    class_counts = df['dx'].value_counts()
    plt.figure(figsize=(10, 5))
    ax = sns.barplot(x=class_counts.index, y=class_counts.values, palette="viridis")
    plt.title("HAM10000 Class Frequency Distribution", fontsize=14, weight="bold")
    plt.xlabel("Disease Code", fontsize=12)
    plt.ylabel("Number of Samples", fontsize=12)
    for p in ax.patches:
        ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', xytext=(0, 5), textcoords='offset points')
    plt.tight_layout()
    plt.show()

## 3. Patient Demographics & Localization
Analyzes lesion localization across anatomical sites and patient age distribution.

In [ ]:
if 'df' in locals():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    # Age distribution
    sns.histplot(df['age'].dropna(), bins=18, kde=True, ax=ax1, color="skyblue")
    ax1.set_title("Patient Age Distribution", fontsize=13, weight="bold")
    ax1.set_xlabel("Age")

    # Anatomical site
    loc_counts = df['localization'].value_counts()
    sns.barplot(x=loc_counts.values, y=loc_counts.index, ax=ax2, palette="magma")
    ax2.set_title("Lesion Anatomical Site Localization", fontsize=13, weight="bold")
    ax2.set_xlabel("Count")

    plt.tight_layout()
    plt.show()